# Baseline eval on Colab — `post-training-lab`

Thin launcher. Clones the repo, installs the `[eval]` extra, runs `make eval-baseline`,
produces `results/metrics.json` with the un-tuned Qwen2.5-0.5B-Instruct row.

**Before you start**
1. Runtime → Change runtime type → **GPU** (T4 free; L4/A100 on Pro).
2. In the left sidebar, open **Secrets** and add:
   - `HF_TOKEN` — your Hugging Face read token (`huggingface.co/settings/tokens`).
   - `WANDB_API_KEY` — optional, only if you want runs to log to W&B.
3. Update `GITHUB_USER` in the clone cell below to your handle.

Reference: `PROJECT.md` §3 (Colab bootstrap), §4.3 (HF cache), §5.4 (smoke-test discipline).

## 1. GPU sanity check

If `nvidia-smi` fails or shows no GPU, the runtime is CPU-only — switch it before continuing.

In [ ]:
!nvidia-smi

## 2. Clone the repo

Set `GITHUB_USER` to your GitHub handle. If the repo is private, generate a fine-grained
personal access token with read access and pass it inline:
`git clone https://<TOKEN>@github.com/<user>/post-training-lab.git`.

In [ ]:
GITHUB_USER = "your-github-handle"   # <-- edit me
REPO        = "post-training-lab"
BRANCH      = "main"

import os, shutil
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone --depth 1 --branch {BRANCH} https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!git log -1 --oneline

## 3. (Optional) Mount Google Drive for the HF cache

Colab wipes the runtime when it disconnects. Pointing `HF_HOME` at Drive means the model
weights (~1 GB) and the lm-eval task data download **once**, not every session. Skip this
cell on the first run if you'd rather not mount Drive.

In [ ]:
USE_DRIVE_CACHE = True

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["HF_HOME"] = "/content/drive/MyDrive/hf-cache"
    os.makedirs(os.environ["HF_HOME"], exist_ok=True)
    print("HF_HOME =", os.environ["HF_HOME"])
else:
    print("Using ephemeral runtime cache — re-downloads on disconnect.")

## 4. Install the project

Colab doesn't ship `uv`, so we fall back to plain `pip install -e .[eval]` — the Colab
bootstrap pattern PROJECT.md §3 commits to. This pulls in `lm-eval`, `transformers`, `trl`,
`peft`, etc. First install on a fresh runtime is 2–5 minutes.

In [ ]:
!pip install -q -e .[eval]

## 5. Inject secrets

Reads from Colab's Secrets panel (left sidebar key icon). Nothing is printed.

In [ ]:
from google.colab import userdata

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key:14s} loaded")
    except Exception:
        print(f"{key:14s} not set (optional)")

# huggingface_hub picks up HF_TOKEN automatically when present.
# wandb mode -> offline if no API key, to avoid an interactive prompt mid-eval.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_MODE"] = "disabled"

## 6. (T4 only) Switch dtype to fp16

Skip this cell on **A100 / L4 / H100** — they have native bf16. Run it on **T4**:
Turing emulates bf16 in software and runs roughly half-speed. We swap the resolved
config's dtype before launching the eval.

In [ ]:
import subprocess
gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip()
print("GPU:", gpu_name)

if "T4" in gpu_name:
    # Override on base.yaml — baseline.yaml inherits from it.
    import re, pathlib
    p = pathlib.Path("configs/base.yaml")
    text = p.read_text()
    new = re.sub(r"dtype:\s*bfloat16", "dtype: float16", text)
    p.write_text(new)
    print("Patched configs/base.yaml -> dtype: float16 (T4 fp16 is faster than emulated bf16).")
else:
    print("Native bf16 GPU — leaving config untouched.")

## 7. Smoke test (10 samples per task)

Run **before** the full eval. If anything is broken — bad dataset args, OOM, tokenizer
mismatch — it fails here in 2–3 minutes instead of an hour into IFEval. PROJECT.md §5.4:
*"Never launch a paid GPU on code you haven't run 50 steps of on free tier."*

In [ ]:
# RUNNER= overrides the Makefile's `uv run` default. Colab has no uv (PROJECT.md
# §3) — we use the system Python that `pip install -e .[eval]` populated above.
# Without this, `uv run` would spin up a fresh .venv with only the core deps,
# and lm_eval (in the [eval] extra) would be missing.
!make eval-smoke RUNNER=

## 8. Full baseline eval

All four tasks at their configured shots/limits (see `configs/base.yaml`):

| Task | Few-shot | Limit |
|---|---|---|
| MMLU | 5 | 1000 |
| GSM8K | 8 | full |
| TruthfulQA MC2 | 0 | full |
| IFEval | 0 | full |

Expected wall-clock on a free T4: **~30–60 min** total, IFEval dominates.

In [ ]:
# See cell 14 for why RUNNER= is needed on Colab.
!make eval-baseline RUNNER=

## 9. Inspect the results

In [ ]:
import json, pathlib
doc = json.loads(pathlib.Path("results/metrics.json").read_text())
print(json.dumps(doc, indent=2))

## 10. Download `metrics.json` so you can commit it locally

Easier than wiring git push from Colab. Drop the downloaded file into
`results/metrics.json` in your local checkout and commit.

In [ ]:
from google.colab import files
files.download("results/metrics.json")